# RAG on Cloudflare — Two Paths

This notebook shows both ways to do RAG:

1. **`KnowledgeBase`** (managed) — Uses AI Search with hybrid search, BM25, reranking. Cloudflare handles everything.
2. **`DIYKnowledgeBase`** — Uses Vectorize + Workers AI. You control chunking, embedding, reranking.

```
Path 1 (managed): Your docs → AI Search → hybrid search + reranking → answer
Path 2 (DIY):     Your docs → chunk → embed → Vectorize → search → rerank → answer
```

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

## Path 1: KnowledgeBase (managed RAG)

**Prereqs:** Create an AI Search instance in the dashboard (AI → AI Search), point it at an R2 bucket or website.

API token needs: AI Search Edit + AI Search Run

In [ ]:
from pydantic_ai_cloudflare import KnowledgeBase

# Create a KnowledgeBase pointing at your AI Search instance
kb = KnowledgeBase(
    "my-docs",
    retrieval_type="hybrid",  # semantic + BM25 keyword search
    reranking=True,            # cross-encoder reranking
    context_expansion=1,       # include surrounding chunks
)

# Search: returns ranked chunks with scores
results = await kb.search("How does caching work?")
for r in results:
    print(f"[{r.get('score', '')}] {r.get('text', '')[:100]}")

# Ask: returns AI-generated answer with citations
answer = await kb.ask("How does caching work?")
print(f"\nAnswer: {answer}")

---

## Path 2: DIYKnowledgeBase

**Prereq:** `npx wrangler vectorize create my-vectors --dimensions 768 --metric cosine`

API token needs: Workers AI Read + Vectorize Edit + Browser Rendering Edit

### Step 1: Ingest content

Pass URLs (fetched via Browser Run) or raw text. The library chunks, embeds, and stores automatically.

In [ ]:
from pydantic_ai_cloudflare import DIYKnowledgeBase

kb = DIYKnowledgeBase(
    "my-vectors",
    chunk_size=800,           # chars per chunk
    chunk_overlap=200,        # overlap between chunks
    reranker_model="@cf/baai/bge-reranker-base",
)

# Ingest from URLs (Browser Run fetches + renders JS)
stats = await kb.ingest([
    "https://developers.cloudflare.com/workers-ai/",
    "https://developers.cloudflare.com/vectorize/",
])
print(f"Ingested: {stats['chunks_created']} chunks, {stats['vectors_stored']} vectors stored")

### Ingest raw text (local documents)

In [ ]:
# You can also ingest raw text directly
stats = await kb.ingest([
    "Cloudflare Workers AI provides free LLM inference on 20+ models.",
    "Vectorize is a vector database for semantic search and RAG.",
    "Browser Run provides headless Chrome on the edge for web scraping.",
], metadata={"source": "internal-notes"})
print(f"Ingested: {stats['chunks_created']} chunks")

### Step 2: Search with reranking

In [ ]:
# Search: embed query → Vectorize → rerank with cross-encoder
results = await kb.search("What is Workers AI?", rerank=True)

for r in results:
    score = r.get('rerank_score', r.get('vector_score', 0))
    print(f"  [{score:.3f}] {r['text'][:80]}...")

### How the chunker works

The built-in chunker splits on natural boundaries (paragraphs → sentences → words) with overlap. No external dependencies.

In [2]:
from pydantic_ai_cloudflare.knowledge import chunk_text

text = """
Cloudflare Workers AI lets you run machine learning models on Cloudflare's global network.
It supports text generation, embeddings, image classification, and more.

Workers AI is integrated with the rest of Cloudflare's developer platform. You can use it
with Workers, Pages, R2, D1, and other services. Models run on GPUs deployed across
Cloudflare's network, close to users.

Key features include:
- Over 20 open-source models available
- Pay-per-use pricing with a generous free tier
- OpenAI-compatible API for easy migration
- Built-in AI Gateway for logging and analytics
- Support for fine-tuning with LoRA adapters
""".strip()

chunks = chunk_text(text, chunk_size=200, overlap=50)
print(f"{len(text)} chars \u2192 {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"\nChunk {i} ({len(c)} chars):")
    print(f"  {c[:60]}...")

624 chars → 5 chunks

Chunk 0 (163 chars):
  Cloudflare Workers AI lets you run machine learning models o...
Chunk 1 (139 chars):
  ion, embeddings, image classification, and more. Workers AI...
Chunk 2 (171 chars):
  f Cloudflare's developer platform. You can use it with Worke...
Chunk 3 (158 chars):
  yed across Cloudflare's network, close to users. Key featur...
Chunk 4 (185 chars):
  e - Pay-per-use pricing with a generous free tier - OpenAI-c...


### Embeddings

In [3]:
from pydantic_ai_cloudflare import CloudflareEmbeddingModel

model = CloudflareEmbeddingModel(gateway_id=None)
result = await model.embed("What is Workers AI?", input_type="query")
print(f"Dimensions: {len(result.embeddings[0])}")
print(f"First 3 values: {[round(v, 3) for v in result.embeddings[0][:3]]}")

Dimensions: 768
First 3 values: [0.051, 0.003, -0.011]


## Retrieval pipeline

```
DIYKnowledgeBase pipeline:
  Query → Embed (bge-base) → Vectorize (top 20) → Rerank (bge-reranker) → Top 5

KnowledgeBase pipeline (AI Search):
  Query → Rewrite (LLM) → Embed → Vector search + BM25 → RRF fusion → Rerank → Top K
```

AI Search handles everything in one API call. DIYKnowledgeBase gives you full control.